# Week 5

### Lab 01

In [14]:
import numpy as np
T = np.eye(4)# Start with identity matrix
T[0, 3] = 0.25# x translation: 25 cm
T[1, 3] = 0.10# y translation: 10 cm
T[2, 3] = 0.40# z translation: 40 cm
# T[:3, :3] holds the rotation matrix (identity = no rotation)
print(T)

[[          1           0           0        0.25]
 [          0           1           0         0.1]
 [          0           0           1         0.4]
 [          0           0           0           1]]


### Lab 02

In [2]:
pose = {
"x":0.32,
"y":-0.05,
"z":0.41,
"roll":0.10,
# meters
# radians
"pitch": -0.02,
"yaw":
1.57
# ~90 degrees
}
print(f"Position: ({pose['x']}, {pose['y']}, {pose['z']})")
print(f"Orientation (RPY): ({pose['roll']}, {pose['pitch']}, {pose['yaw']})")

Position: (0.32, -0.05, 0.41)
Orientation (RPY): (0.1, -0.02, 1.57)


### Lab 03

In [8]:
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")

img = cv2.imread("scene.jpg")
if img is None:
    print("Error: Could not load scene.jpg")
    exit()

result = model(img)


0: 448x640 8 persons, 1 backpack, 4.1ms
Speed: 1.0ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)


In [9]:
for r in result:
    for box in r.boxes:         
        cls_id = int(box.cls[0])        
        conf   = float(box.conf[0])     
        
        # 1. Convert the box corners directly to Python integers
        x1, y1, x2, y2 = map(int, box.xyxy[0])   
        
        label = model.names[cls_id]     
        print(f"{label}: {conf:.2f} box=[{x1},{y1},{x2},{y2}]")
        
        # 2. Calculate centroid (using integer division //)
        u = (x1 + x2) // 2
        v = (y1 + y2) // 2
        print(f"Centroid: u={u}, v={v}")
        
        # 3. Crop the ROI (indented inside the loop)
        roi = img[y1:y2, x1:x2]
        
        # You can now process each 'roi' individually here
        # cv2.imshow(f"Cropped {label}", roi)

person: 0.92 box=[441,93,524,333]
Centroid: u=482, v=213
person: 0.90 box=[111,102,187,312]
Centroid: u=149, v=207
person: 0.88 box=[276,107,367,344]
Centroid: u=321, v=225
person: 0.86 box=[48,105,124,322]
Centroid: u=86, v=213
person: 0.86 box=[227,119,283,297]
Centroid: u=255, v=208
person: 0.77 box=[389,115,443,245]
Centroid: u=416, v=180
person: 0.70 box=[190,132,238,262]
Centroid: u=214, v=197
backpack: 0.38 box=[389,132,419,181]
Centroid: u=404, v=156
person: 0.33 box=[27,135,53,214]
Centroid: u=40, v=174


### Lab 04

In [11]:
import cv2
import numpy as np

# 1. 3D Model Points (Coordinates of features in the object's own 3D space)
# Let's assume a simple 4-point square object (e.g., a tracking tag)
object_points = np.array([
    [0.0, 0.0, 0.0],      # Point 1 (top-left)
    [100.0, 0.0, 0.0],    # Point 2 (top-right)
    [100.0, 100.0, 0.0],  # Point 3 (bottom-right)
    [0.0, 100.0, 0.0]     # Point 4 (bottom-left)
], dtype=np.float32)

# 2. 2D Image Points (Where those exact 3D points ended up in your pixel image)
# These usually come from your detector (like a chessboard finder or YOLO centroid)
image_points = np.array([
    [230, 145],  # Point 1 in pixels
    [385, 150],  # Point 2 in pixels
    [390, 295],  # Point 3 in pixels
    [225, 290]   # Point 4 in pixels
], dtype=np.float32)

# 3. Camera Matrix (Intrinsic parameters of your specific camera lens/sensor)
# If you haven't calibrated your camera, you can approximate it based on image resolution
focal_length = 800  # Approximated focal length in pixels
cx, cy = 320, 240   # Image center (assuming a 640x480 image)
camera_matrix = np.array([
    [focal_length, 0, cx],
    [0, focal_length, cy],
    [0, 0, 1]
], dtype=np.float32)

# 4. Distortion Coefficients (Lens warping parameters)
# Set to zero if you assume a perfectly undistorted pinhole camera
dist_coeffs = np.zeros((4, 1), dtype=np.float32) 

# --- NOW YOU CAN RUN SOLVEPNP ---
ret, rvec, tvec = cv2.solvePnP(
    object_points,
    image_points,
    camera_matrix,
    dist_coeffs
)

# Convert rotation vector to a 3x3 rotation matrix
R, _ = cv2.Rodrigues(rvec)

print("Rotation Matrix R:\n", R)
print("Translation Vector tvec:\n", tvec)

Rotation Matrix R:
 [[    0.99652   0.0049896    -0.08323]
 [   0.028313     0.91865     0.39407]
 [   0.078425    -0.39505     0.91531]]
Translation Vector tvec:
 [[    -57.501]
 [    -61.648]
 [     516.46]]


In [12]:
ret, rvec, tvec = cv2.solvePnP(
object_points,# Nx3 array of 3D model points
image_points,# Nx2 array of 2D image points
camera_matrix,# 3x3 intrinsic matrix K
dist_coeffs# lens distortion coefficients
)
# Convert rotation vector to rotation matrix
R, _ = cv2.Rodrigues(rvec)

### Lab 05

In [16]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

# ==========================================
# 1. INITIALIZATION & CALIBRATION SETUP
# ==========================================

# Initialize YOLOv8 Model
model = YOLO("yolov8n.pt") 

# Define standard 3D Model Points (e.g., a 10cm x 10cm square target object)
# Coordinates are in meters relative to the object's own origin
object_points = np.array([
    [-0.05,  0.05, 0.0],  # Top-left
    [ 0.05,  0.05, 0.0],  # Top-right
    [ 0.05, -0.05, 0.0],  # Bottom-right
    [-0.05, -0.05, 0.0]   # Bottom-left
], dtype=np.float32)

# Camera Intrinsic Matrix (K) - Replace with your actual calibration data
focal_length = 800  
cx, cy = 320, 240   
camera_matrix = np.array([
    [focal_length, 0, cx],
    [0, focal_length, cy],
    [0, 0, 1]
], dtype=np.float32)

# Distortion Coefficients
dist_coeffs = np.zeros((4, 1), dtype=np.float32) 

# Hand-Eye Calibration Matrix: T_robot_camera (Camera position relative to Robot Base)
# X = +0.25m, Y = +0.10m, Z = +0.40m
T_robot_camera = np.eye(4, dtype=np.float64)
T_robot_camera[0, 3] = 0.25  
T_robot_camera[1, 3] = 0.10  
T_robot_camera[2, 3] = 0.40  


# ==========================================
# 2. IMAGE INGESTION & OBJECT DETECTION
# ==========================================

image_path = "scene.jpg"
img = cv2.imread(image_path)

if img is None:
    print(f"Error: Could not load image from '{image_path}'. Check file path.")
    exit()

# Run Object Detection
results = model(img)

# Process detected objects
for r in results:
    for box in r.boxes:         
        cls_id = int(box.cls[0])        
        conf   = float(box.conf[0])     
        label  = model.names[cls_id]
        
        # We look for a specific target, e.g., a "cup" or "sports ball" (using 'person' here as an example)
        if label != "person" or conf < 0.5:
            continue
            
        print(f"\n--- Processing Target: {label} (Conf: {conf:.2f}) ---")
        
        # Convert bounding box tensors safely to integer coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])   
        
        # Isolate Region of Interest (ROI)
        roi = img[y1:y2, x1:x2]
        
        # Calculate 2D Centroid pixel
        u = (x1 + x2) // 2
        v = (y1 + y2) // 2
        print(f"2D Image Space Centroid: u={u}, v={v}")

        # ==========================================
        # 3. POSE ESTIMATION (solvePnP)
        # ==========================================
        
        # Map your bounding box corners to 2D image points for PnP
        image_points = np.array([
            [x1, y1],  # Top-left
            [x2, y1],  # Top-right
            [x2, y2],  # Bottom-right
            [x1, y2]   # Bottom-left
], dtype=np.float32)
        
        # Compute 3D Translation and Rotation vectors relative to Camera
        success, rvec, tvec = cv2.solvePnP(
            object_points, image_points, camera_matrix, dist_coeffs
        )
        
        if not success:
            print("Failed to compute 3D pose via solvePnP.")
            continue
            
        # Convert Rotation Vector to a 3x3 Matrix
        R_camera, _ = cv2.Rodrigues(rvec)
        
        # Construct the 4x4 Homogeneous Transformation Matrix (pose_in_camera)
        pose_in_camera = np.eye(4, dtype=np.float64)
        pose_in_camera[:3, :3] = R_camera
        pose_in_camera[:3, 3] = tvec.squeeze()
        
        print(f"Pose in Camera Frame (XYZ meters):\n{pose_in_camera[:3, 3]}")

        # ==========================================
        # 4. COORDINATE SPACE TRANSFORMATION
        # ==========================================
        
        # Transform pose from Camera Frame to Robot Base Frame
        pose_in_robot = T_robot_camera @ pose_in_camera
        
        print(f"Target Pose in Robot Base Frame:\n{pose_in_robot}")
        
        # ==========================================
        # 5. ROBOT EXECUTION PREPARATION
        # ==========================================
        
        # Extract precise target values for your IK solver
        target_xyz = pose_in_robot[:3, 3]
        target_rotation_matrix = pose_in_robot[:3, :3]
        
        print(f"Sending coordinates to Robot Motion Planner...")
        print(f"-> Move Tool Center Point to: X={target_xyz[0]:.3f}m, Y={target_xyz[1]:.3f}m, Z={target_xyz[2]:.3f}m")
        
        # robot_arm.move_to(pose_in_robot)


0: 448x640 8 persons, 1 backpack, 4.1ms
Speed: 1.0ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)

--- Processing Target: person (Conf: 0.92) ---
2D Image Space Centroid: u=482, v=213
Pose in Camera Frame (XYZ meters):
[   0.073226   -0.011568      0.3492]
Target Pose in Robot Base Frame:
[[    0.54269  -0.0059787     0.83991     0.32323]
 [   -0.02993    -0.99948    0.012224    0.088432]
 [     0.8394   -0.031772    -0.54259      0.7492]
 [          0           0           0           1]]
Sending coordinates to Robot Motion Planner...
-> Move Tool Center Point to: X=0.323m, Y=0.088m, Z=0.749m

--- Processing Target: person (Conf: 0.90) ---
2D Image Space Centroid: u=149, v=207
Pose in Camera Frame (XYZ meters):
[  -0.086472   -0.016061     0.39497]
Target Pose in Robot Base Frame:
[[    0.56001   0.0076298    -0.82845     0.16353]
 [   0.035609    -0.99926    0.014868    0.083939]
 [   -0.82772   -0.037827    -0.55986     0.79497]
 [          0  